# Audio Basics for TTS

This notebook covers fundamental audio concepts needed for Text-to-Speech:

1. What is digital audio?
2. Sample rate and bit depth
3. Loading and playing audio
4. Basic audio visualization
5. Audio normalization

In [ ]:
# Install dependencies if needed
# !pip install numpy librosa matplotlib soundfile IPython

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# Set up plotting
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. What is Digital Audio?

Digital audio is a representation of sound as a sequence of numbers (samples).

Key concepts:
- **Sample**: A single measurement of air pressure at a point in time
- **Sample Rate**: How many samples per second (Hz)
- **Amplitude**: The loudness at each sample point

In [ ]:
# Create a simple sine wave - the most basic sound
sample_rate = 22050  # samples per second (common for TTS)
duration = 1.0  # seconds
frequency = 440  # Hz (A4 note)

# Generate time array
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# Generate sine wave
amplitude = 0.5
audio = amplitude * np.sin(2 * np.pi * frequency * t)

print(f"Sample rate: {sample_rate} Hz")
print(f"Duration: {duration} seconds")
print(f"Total samples: {len(audio)}")
print(f"Audio shape: {audio.shape}")
print(f"Value range: [{audio.min():.2f}, {audio.max():.2f}]")

In [ ]:
# Visualize the waveform
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Full waveform
axes[0].plot(t, audio)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Full Waveform (1 second)')

# Zoomed in - first few cycles
zoom_samples = 200
axes[1].plot(t[:zoom_samples], audio[:zoom_samples], 'b-')
axes[1].plot(t[:zoom_samples], audio[:zoom_samples], 'r.', markersize=3)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Zoomed View (showing individual samples)')

plt.tight_layout()
plt.show()

In [ ]:
# Play the audio
display(Audio(audio, rate=sample_rate))

## 2. Sample Rate

Sample rate determines the highest frequency that can be represented.

**Nyquist theorem**: Maximum frequency = Sample Rate / 2

Common sample rates:
- 8000 Hz: Telephone quality
- 16000 Hz: Wideband speech (common for ASR)
- 22050 Hz: Common for TTS
- 44100 Hz: CD quality
- 48000 Hz: Professional audio

In [ ]:
# Compare different sample rates
sample_rates = [8000, 16000, 22050, 44100]

fig, axes = plt.subplots(len(sample_rates), 1, figsize=(12, 8))

for i, sr in enumerate(sample_rates):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    audio_sr = amplitude * np.sin(2 * np.pi * frequency * t)
    
    axes[i].plot(t[:100], audio_sr[:100], 'b-o', markersize=3)
    axes[i].set_title(f'Sample Rate: {sr} Hz (Max frequency: {sr/2} Hz)')
    axes[i].set_ylabel('Amplitude')

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

## 3. Complex Sounds (Harmonics)

Real sounds (including speech) are made of multiple frequencies.

A musical note has:
- **Fundamental frequency**: The main pitch
- **Harmonics**: Multiples of the fundamental (2x, 3x, 4x...)

In [ ]:
# Create a complex tone with harmonics
def generate_complex_tone(fundamental, harmonics, amplitudes, duration, sample_rate):
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    signal = np.zeros_like(t)
    
    for harmonic, amp in zip(harmonics, amplitudes):
        freq = fundamental * harmonic
        signal += amp * np.sin(2 * np.pi * freq * t)
    
    # Normalize
    signal = signal / np.max(np.abs(signal)) * 0.8
    return signal, t

# Generate a vowel-like sound
fundamental = 150  # Hz (typical male voice fundamental)
harmonics = [1, 2, 3, 4, 5, 6]
amplitudes = [1.0, 0.5, 0.3, 0.2, 0.1, 0.05]

complex_audio, t = generate_complex_tone(
    fundamental, harmonics, amplitudes, 1.0, 22050
)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t[:500], complex_audio[:500])
axes[0].set_title('Complex Tone (with harmonics)')
axes[0].set_ylabel('Amplitude')

# Compare with pure sine wave
pure_sine = 0.8 * np.sin(2 * np.pi * fundamental * t)
axes[1].plot(t[:500], pure_sine[:500])
axes[1].set_title('Pure Sine Wave (no harmonics)')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

print("Complex tone:")
display(Audio(complex_audio, rate=22050))

print("\nPure sine wave:")
display(Audio(pure_sine, rate=22050))

## 4. Loading Real Audio

Let's use librosa to load and analyze real audio files.

In [ ]:
try:
    import librosa
    import librosa.display
    
    # Load a sample audio file (librosa includes example files)
    audio, sr = librosa.load(librosa.ex('trumpet'), sr=22050, duration=5)
    
    print(f"Loaded audio:")
    print(f"  Sample rate: {sr}")
    print(f"  Duration: {len(audio)/sr:.2f} seconds")
    print(f"  Shape: {audio.shape}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 4))
    librosa.display.waveshow(audio, sr=sr, ax=ax)
    ax.set_title('Waveform')
    plt.show()
    
    # Play
    display(Audio(audio, rate=sr))
    
except Exception as e:
    print(f"Could not load librosa example: {e}")
    print("Try: pip install librosa")

## 5. Audio Normalization

Normalization ensures consistent volume levels.

In [ ]:
def normalize_audio(audio, target_rms_db=-20):
    """
    Normalize audio to a target RMS level.
    
    Args:
        audio: Input audio array
        target_rms_db: Target RMS level in dB
    
    Returns:
        Normalized audio
    """
    # Calculate current RMS
    rms = np.sqrt(np.mean(audio ** 2))
    
    if rms == 0:
        return audio
    
    # Calculate current dB
    current_db = 20 * np.log10(rms)
    
    # Calculate gain
    gain_db = target_rms_db - current_db
    gain = 10 ** (gain_db / 20)
    
    # Apply gain
    normalized = audio * gain
    
    return normalized

# Create audio with varying volume
t = np.linspace(0, 3, 3 * 22050, endpoint=False)
envelope = np.concatenate([
    np.ones(22050) * 0.1,   # Quiet
    np.ones(22050) * 0.5,   # Medium
    np.ones(22050) * 1.0,   # Loud
])
varying_audio = envelope * np.sin(2 * np.pi * 440 * t)

# Normalize
normalized = normalize_audio(varying_audio)

# Compare
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t, varying_audio)
axes[0].set_title('Original (varying volume)')
axes[0].set_ylabel('Amplitude')

axes[1].plot(t, normalized)
axes[1].set_title('Normalized')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

## Summary

Key takeaways:

1. **Digital audio** is a sequence of amplitude samples
2. **Sample rate** determines frequency range (22050 Hz is common for TTS)
3. **Real sounds** contain multiple frequencies (harmonics)
4. **Normalization** ensures consistent volume

Next: In the spectrograms notebook, we'll learn how to visualize the frequency content of audio.